# 02 — تدريب النموذج على بيانات قرآنية (Fine-tuning)
**Report Chapter Four, Step 12, and Table 5.5.** شغّله على GPU: Runtime ← Change runtime type ← T4 GPU.

**ليش:** النماذج الجاهزة دُرّبت على نصوص عامة، وما تفرّق بين معاني الجذر الواحد في القرآن (مثل «نفق»:
النفاق والإنفاق). نكمّل تدريب `paraphrase-multilingual-mpnet-base-v2` على أزواج (تفسير ← آية) من التفسير
الميسر، فيتعلم أن يقرّب وصف المعنى بلغة عادية من اللفظ القرآني الذي يعبّر عنه.

**تطوّر البيانات:**
- النسخة الأولى: 6236 زوجًا (فقرة التفسير كاملة ← الآية). تحسّن «النفاق» كثيرًا، لكن تراجعت أسئلة قصيرة
  مثل «الطهارة» و«الحياء»، لأن التدريب كان على فقرات طويلة والأسئلة الحقيقية قصيرة.
- النسخة النهائية (`finetune_pairs_v2.csv`): 16929 زوجًا = 6236 فقرة كاملة + 10693 عبارة قصيرة (3-14 كلمة)
  من نفس التفسير تشترك في جذر مع كلمة من الآية، فتشبه شكل السؤال الحقيقي.

**فصل التدريب عن التقييم:** أسئلة التقييم نفسها لم تدخل التدريب.

**الملف المطلوب رفعه:** `finetune_pairs_v2.csv` بأعمدة `anchor, verse_text, anchor_type`.

In [ ]:
!pip -q install sentence-transformers nltk streamlit openpyxl
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "غير متاح — التدريب سيكون بطيئًا")

## 1) أزواج التدريب

In [ ]:
import pandas as pd
from sentence_transformers import InputExample

pairs = pd.read_csv("finetune_pairs_v2.csv").sample(frac=1.0, random_state=42).reset_index(drop=True)
print("عدد الأزواج:", len(pairs))
print(pairs["anchor_type"].value_counts())
train_examples = [InputExample(texts=[r.anchor, r.verse_text]) for r in pairs.itertuples()]

## 2) التدريب
- الخسارة: `MultipleNegativesRankingLoss` — كل زوج آخر في نفس الدفعة يُعامل كمثال سلبي.
- الإعدادات النهائية: دفعة 32، معدل تعلّم 5e-5، خمس دورات (epochs)، أقصى طول 128 رمزًا. قرابة ساعة على T4.
- الدفعة 64 سببت نفاد ذاكرة GPU مع الفقرات الطويلة، فخُفّضت إلى 32.

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

BASE_MODEL = "paraphrase-multilingual-mpnet-base-v2"
BATCH_SIZE, EPOCHS, LR = 32, 5, 5e-5

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = 128
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
loss = losses.MultipleNegativesRankingLoss(model)
model.fit(train_objectives=[(loader, loss)], epochs=EPOCHS, optimizer_params={"lr": LR},
          warmup_steps=int(0.1 * len(loader) * EPOCHS), show_progress_bar=True,
          output_path="quran_finetuned_mpnet")
print("حُفظ النموذج في quran_finetuned_mpnet/")

## 3) التقييم: قبل التدريب وبعده، بنفس كود الموقع
نستخدم `app.py` من المستودع (نفس التطبيع ونفس الترتيب الهجين) ومقياس التقييم (16 موضوعًا). الفرق الوحيد
بين العمودين هو النموذج.

In [ ]:
!git clone -q https://github.com/f6pmt92g5t-ops/quran-semantic-search
import os, sys
os.chdir("/content/quran-semantic-search")
sys.path.insert(0, ".")
sys.path.insert(0, "evaluation")

import numpy as np
import app
from colab_compare_models import evaluate   # نفس دالة التقييم المستخدمة في مقارنة النماذج

D = app.load_data()
seg_texts = pd.read_csv("segments.csv")["text"].apply(app.normalize_arabic).tolist()
summary = []
for label, path in [("قبل: mpnet الأصلي", BASE_MODEL), ("بعد: mpnet المدرَّب", "/content/quran_finetuned_mpnet")]:
    m = SentenceTransformer(path)
    emb = m.encode(seg_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    D.emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

    def encode_query(q, m=m):
        v = m.encode(app.normalize_arabic(q)).astype(np.float32)
        return v / (np.linalg.norm(v) + 1e-12)

    res = evaluate(D, encode_query)
    summary.append({"النموذج": label, **res.drop(columns=["query", "hyb top10"]).mean().round(3).to_dict()})

pd.DataFrame(summary)

## 4) النشر: رفع النموذج على Hugging Face وإعادة تشفير الأجزاء
الموقع يحمّل النموذج بالاسم `Amer-Surur1/quran-finetuned-mpnet`، ويحتاج `segment_embeddings.npy` مشفّرًا بنفس
النموذج (وإلا صار البحث يقارن فضاءين مختلفين).

**الأمان:** لا تكتب رمز Hugging Face داخل النوتبوك. الخلية تطلب الرمز في مربع مخفي ولا يُحفظ في الملف.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
finetuned = SentenceTransformer("/content/quran_finetuned_mpnet")
finetuned.push_to_hub("Amer-Surur1/quran-finetuned-mpnet")

emb = finetuned.encode(seg_texts, batch_size=64, show_progress_bar=True).astype(np.float16)
assert emb.shape == (10675, 768)
np.save("/content/segment_embeddings.npy", emb)

from google.colab import files
files.download("/content/segment_embeddings.npy")